![Cloud-First](../image/CloudFirst.png)

# SIT742: Modern Data Science
**(Module 03: Big Data)**

**Session 3E: Data Acquisition II**

---

- Materials in this module have been developed to support practical learning in modern data science, big data processing, and applied analytics.
- Materials may include adapted or referenced open-source resources. Keep attribution and licence notes where applicable.
- The public notebook collection is available in [SIT742](https://github.com/tulip-lab/sit742).
- If you find any issue or bug in this document, please submit an issue at [SIT742](https://github.com/tulip-lab/sit742/issues).
- Audience: Honours and Master's students using the Deakin SIT742 practical and self-learning materials.

Prepared by the SIT742 Teaching Team.

Maintained through the [TULIP Lab](https://www.tulip.academy) FLIP workflow.

---

<div align="center">

<table>
<thead>
<tr>
<th><strong>Item</strong></th>
<th><strong>Description</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Module context</td>
<td>This notebook is one component of the practical and self-learning materials for Module 03. The full module is normally completed across two two-hour practical sessions, together with the other notebooks listed in the SIT742 repository.</td>
</tr>
<tr>
<td align="left">Environment</td>
<td>Google Colab or local Jupyter with Python, pandas, Matplotlib, and Beautiful Soup</td>
</tr>
<tr>
<td align="left">Main output</td>
<td>Portable pandas, live CSV, and web-table acquisition workflow</td>
</tr>
<tr>
<td align="left">Related assessment</td>
<td>General practical skill development; not directly assessed</td>
</tr>
</tbody>
</table>

</div>

---

**Table of Contents**

- [1. Overview and Learning Goals](#1-overview-and-learning-goals)
- [2. Setup and Data Sources](#2-setup-and-data-sources)
- [3. Background Concepts](#3-background-concepts)
- [4. Guided Examples](#4-guided-examples)
- [5. Practical Exercises](#5-practical-exercises)
- [6. Student Tasks](#6-student-tasks)
- [7. Reflection and References](#7-reflection-and-references)


<a id="1-overview-and-learning-goals"></a>
### 1. Overview and Learning Goals

This session extends data acquisition from local file formats to tabular data, live CSV endpoints, and HTML tables. The emphasis is on using pandas for reliable tabular workflows and on treating live web data as useful but fragile.

By the end of this lab, students should be able to:

1. create and inspect pandas `Series` and `DataFrame` objects;
2. load a public CSV file in both online and local notebook environments;
3. read a bounded live CSV data source and store generated output safely;
4. extract a public web table with Beautiful Soup using Python 3; and
5. explain why live web examples need fallback and validation checks.


<a id="2-setup-and-data-sources"></a>
### 2. Setup and Data Sources

This notebook uses one public SIT742 file, `Melbourne_bike_share.csv`, plus two live public web sources:

- Environment and Climate Change Canada historical weather CSV downloads;
- Wikipedia's page on Australian states and territories.

Live web sources can change. The code below uses small, bounded requests and includes teaching fallbacks so the notebook remains usable if a live source is unavailable.


In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen, urlretrieve
import importlib.util
import subprocess
import sys
import tempfile
import warnings

import pandas as pd
import matplotlib.pyplot as plt

PUBLIC_DATA_BASE_URL = "https://raw.githubusercontent.com/tulip-lab/sit742/develop/Jupyter/data"
EXECUTION_MODE = "online"  # Use "online" for Google Colab; use "local" for a cloned SIT742 repository.
INSTALL_MISSING_PACKAGES = EXECUTION_MODE == "online"

required_files = ["Melbourne_bike_share.csv"]
OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="sit742_m03e_output_"))


def ensure_package(import_name, package_name):
    if importlib.util.find_spec(import_name) is not None:
        return
    if not INSTALL_MISSING_PACKAGES:
        raise ImportError(
            f"{package_name} is required. Install it or switch EXECUTION_MODE to 'online'."
        )
    subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])


ensure_package("bs4", "beautifulsoup4")
from bs4 import BeautifulSoup


def download_public_data(required_files):
    data_dir = Path(tempfile.mkdtemp(prefix="sit742_m03e_data_"))
    downloaded_paths = {}
    for filename in required_files:
        url = f"{PUBLIC_DATA_BASE_URL}/{filename}"
        local_file = data_dir / filename
        urlretrieve(url, local_file)
        downloaded_paths[filename] = local_file
    return data_dir, downloaded_paths


def find_local_data_dir(required_files):
    candidates = [
        Path.cwd() / "Jupyter" / "data",
        Path.cwd() / "data",
        Path.cwd().parent / "data",
        Path.cwd().parent.parent / "Jupyter" / "data",
    ]
    for candidate in candidates:
        if all((candidate / filename).exists() for filename in required_files):
            return candidate
    searched = "\n".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(
        "Could not find the SIT742 public data folder. Searched:\n" + searched
    )


if EXECUTION_MODE == "online":
    DATA_DIR, data_paths_by_file = download_public_data(required_files)
elif EXECUTION_MODE == "local":
    DATA_DIR = find_local_data_dir(required_files)
    data_paths_by_file = {filename: DATA_DIR / filename for filename in required_files}
else:
    raise ValueError('EXECUTION_MODE must be "online" or "local"')

data_paths = {"bike_share": data_paths_by_file["Melbourne_bike_share.csv"]}
DataSet = data_paths["bike_share"]

print("Python version:", sys.version.split()[0])
print("pandas version:", pd.__version__)
print("Execution mode:", EXECUTION_MODE)
print("Data folder:", DATA_DIR)
print("Temporary output folder:", OUTPUT_DIR)


In [ ]:
for label, path in data_paths.items():
    print(f"{label}: {path.name} ({path.stat().st_size} bytes)")


<a id="3-background-concepts"></a>
### 3. Background Concepts

`pandas` provides two core tabular data structures:

- a `Series`, which behaves like a labelled column;
- a `DataFrame`, which behaves like a labelled table made up of one or more series.

Data acquisition often combines several source types. A local or downloaded CSV file is usually stable. A live CSV endpoint is useful for current or historical public data but can change its parameters. An HTML table can be scraped when no structured file is provided, but the page layout may change.


<a id="4-guided-examples"></a>
### 4. Guided Examples

#### 4.1 pandas basics

The two `DataFrame` definitions below produce the same table from two different input structures.


In [ ]:
df_1 = pd.DataFrame(
    [["a", "b", "c"], ["d", "e", "f"], ["g", "h", "i"]],
    index=[1, 2, 3],
    columns=["col1", "col2", "col3"],
)

df_2 = pd.DataFrame(
    {
        "col1": ["a", "d", "g"],
        "col2": ["b", "e", "h"],
        "col3": ["c", "f", "i"],
    },
    index=[1, 2, 3],
)

print("The two data frames are identical:", df_1.equals(df_2))
df_1


In [ ]:
print("Index:", df_1.index.tolist())
print("Columns:", df_1.columns.tolist())
print("Values:\n", df_1.values)
print("Head:\n", df_1.head(2))
print("Tail:\n", df_1.tail(2))
print("Description:\n", df_1.describe(include="all"))


#### 4.2 Loading public CSV data


In [ ]:
csvdf = pd.read_csv(data_paths["bike_share"])

print("Loaded rows and columns:", csvdf.shape)
print("Columns:", csvdf.columns.tolist())
csvdf.head()


In [ ]:
print("Unique station IDs:", csvdf["ID"].nunique())
print(csvdf.describe(include="all"))

bike_by_id = csvdf.set_index("ID")
print("Index name:", bike_by_id.index.name)
bike_by_id.head()


In [ ]:
bike_available = bike_by_id[["Featurename", "NBBikes", "NBEmptydoc", "UploadDate"]].copy()
bike_available["TotalDocks"] = bike_available["NBBikes"] + bike_available["NBEmptydoc"]
bike_available["PercentAvailable"] = (
    bike_available["NBBikes"] / bike_available["TotalDocks"] * 100
).round(1)

bike_available.sort_values("PercentAvailable", ascending=False).head()


#### 4.3 Reading a bounded live weather CSV endpoint

The next example reads hourly weather observations from Environment and Climate Change Canada. To keep the notebook responsive, it requests only a few months rather than a full year. If the live download is unavailable, a small teaching sample is used so the rest of the workflow still runs.


In [ ]:
WEATHER_MODE = "live"  # Use "sample" to skip the live web request.

WEATHER_URL_TEMPLATE = (
    "https://climate.weather.gc.ca/climate_data/bulk_data_e.html"
    "?format=csv&stationID=5415&Year={year}&Month={month}&Day=14"
    "&timeframe=1&submit=Download+Data"
)


def sample_weather_month(year, month):
    records = [
        {
            "Date/Time (LST)": f"{year}-{month:02d}-01 00:00",
            "Temp (C)": -3.2 + month,
            "Dew Point Temp (C)": -7.0 + month,
            "Rel Hum (%)": 72,
            "Wind Spd (km/h)": 15,
            "Weather": "Clear",
        },
        {
            "Date/Time (LST)": f"{year}-{month:02d}-01 12:00",
            "Temp (C)": 1.4 + month,
            "Dew Point Temp (C)": -2.0 + month,
            "Rel Hum (%)": 64,
            "Wind Spd (km/h)": 20,
            "Weather": "Cloudy",
        },
    ]
    frame = pd.DataFrame(records)
    frame["Date/Time (LST)"] = pd.to_datetime(frame["Date/Time (LST)"])
    return frame.set_index("Date/Time (LST)")


def normalise_weather_columns(frame):
    frame = frame.dropna(axis=1, how="all").copy()
    frame.columns = (
        frame.columns.str.replace("°", "", regex=False)
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )
    return frame


def download_weather_month(year, month, mode=WEATHER_MODE):
    if mode != "live":
        return sample_weather_month(year, month)

    url = WEATHER_URL_TEMPLATE.format(year=year, month=month)
    try:
        frame = pd.read_csv(
            url,
            index_col="Date/Time (LST)",
            parse_dates=True,
            encoding="utf-8-sig",
        )
        frame = normalise_weather_columns(frame)
        if frame.empty or "Temp (C)" not in frame.columns:
            raise ValueError("Live weather response did not contain expected columns.")
        return frame
    except Exception as exc:
        print("Live weather download unavailable; using a small teaching sample.")
        print(type(exc).__name__, exc)
        return sample_weather_month(year, month)


weather_mar2012 = download_weather_month(2012, 3)
weather_mar2012.head()


In [ ]:
weather_months = [download_weather_month(2012, month) for month in [1, 2, 3]]
weather_2012 = pd.concat(weather_months).sort_index()

weather_output = OUTPUT_DIR / "weather_2012_q1.csv"
weather_2012.to_csv(weather_output)

print("Rows:", len(weather_2012))
print("Columns:", weather_2012.columns.tolist()[:10])
print("Saved weather file:", weather_output)
weather_2012[["Temp (C)", "Rel Hum (%)", "Wind Spd (km/h)", "Weather"]].head()


In [ ]:
temperature = weather_2012["Temp (C)"].astype(float)
relative_humidity = weather_2012["Rel Hum (%)"].astype(float)
wind_speed = weather_2012["Wind Spd (km/h)"].astype(float)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
temperature.plot(ax=axes[0], title="Temperature over time")
axes[0].set_ylabel("Temp (C)")

axes[1].scatter(temperature, relative_humidity, alpha=0.6)
axes[1].set_xlabel("Temp (C)")
axes[1].set_ylabel("Relative humidity (%)")
axes[1].set_title("Temperature and humidity")
plt.show()

weather_2012[["Temp (C)", "Rel Hum (%)", "Wind Spd (km/h)"]].describe()


#### 4.4 Extracting a public web table with Beautiful Soup

This example extracts a table from Wikipedia. The code searches for a `wikitable` that contains state/territory and capital information rather than assuming a fixed table number.


In [ ]:
WIKI_MODE = "live"  # Use "sample" to skip the live web request.
WIKI_URL = "https://en.wikipedia.org/wiki/States_and_territories_of_Australia"


def sample_au_states():
    return pd.DataFrame(
        [
            {"State or territory": "New South Wales", "Abbrev.": "NSW", "Capital": "Sydney"},
            {"State or territory": "Victoria", "Abbrev.": "Vic.", "Capital": "Melbourne"},
            {"State or territory": "Queensland", "Abbrev.": "Qld", "Capital": "Brisbane"},
        ]
    )


def fetch_html(url):
    request = Request(url, headers={"User-Agent": "SIT742 teaching notebook"})
    with urlopen(request, timeout=20) as response:
        return response.read()


def table_headers(table):
    return [cell.get_text(" ", strip=True) for cell in table.find_all("th")]


def extract_states_table(mode=WIKI_MODE):
    if mode != "live":
        return sample_au_states()

    try:
        soup = BeautifulSoup(fetch_html(WIKI_URL), "html.parser")
        candidate_tables = soup.find_all(
            "table", class_=lambda value: value and "wikitable" in value
        )
        selected_table = None
        for table in candidate_tables:
            header_text = " ".join(table_headers(table)).lower()
            if "capital" in header_text and ("state" in header_text or "territory" in header_text):
                selected_table = table
                break
        if selected_table is None:
            raise ValueError("Could not find a suitable states and territories table.")

        header_cells = selected_table.find("tr").find_all(["th", "td"])
        headers = [cell.get_text(" ", strip=True) for cell in header_cells]
        rows = []
        for row in selected_table.find_all("tr")[1:]:
            cells = [cell.get_text(" ", strip=True) for cell in row.find_all(["th", "td"])]
            if cells:
                rows.append(cells)

        width = min(len(headers), max(len(row) for row in rows))
        cleaned_rows = [row[:width] for row in rows if len(row) >= width]
        frame = pd.DataFrame(cleaned_rows, columns=headers[:width])
        return frame
    except Exception as exc:
        print("Live Wikipedia extraction unavailable; using a small teaching sample.")
        print(type(exc).__name__, exc)
        return sample_au_states()


df_au = extract_states_table()
print("Rows and columns:", df_au.shape)
df_au.head()


<a id="5-practical-exercises"></a>
### 5. Practical Exercises

Use the checks below to confirm that each acquisition path produced a usable table.


In [ ]:
assert csvdf.shape[0] > 0
assert {"NBBikes", "NBEmptydoc"}.issubset(csvdf.columns)
assert weather_2012.shape[0] > 0
assert "Temp (C)" in weather_2012.columns
assert df_au.shape[0] > 0

print("All acquisition checks passed.")


Try these variations:

1. Set `WEATHER_MODE = "sample"` and rerun the weather cells. What changes?
2. Request a different month from the weather endpoint.
3. Inspect `df_au.columns` and identify which columns would be useful for a map or dashboard.


In [ ]:
# Student workspace
# 1. Try WEATHER_MODE = "sample".
# 2. Request a different month.
# 3. Inspect and select useful columns from df_au.


<a id="6-student-tasks"></a>
### 6. Student Tasks

<div align="center">

<table>
<thead>
<tr>
<th><strong>Task</strong></th>
<th><strong>What you need to do</strong></th>
<th><strong>Why it matters</strong></th>
<th><strong>Expected evidence</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Task 1</td>
<td>Load the Melbourne bike share CSV and create one derived column.</td>
<td>Derived columns turn raw fields into useful indicators.</td>
<td>A table showing the original and derived columns.</td>
</tr>
<tr>
<td align="left">Task 2</td>
<td>Load a bounded live or sample weather dataset and save it into `OUTPUT_DIR`.</td>
<td>Live data workflows need controlled output and fallback handling.</td>
<td>The saved path, row count, and one summary table or plot.</td>
</tr>
<tr>
<td align="left">Task 3</td>
<td>Extract a public HTML table and explain one fragility risk in your approach.</td>
<td>Web tables are useful, but page structure is outside your control.</td>
<td>A DataFrame preview plus a short written note.</td>
</tr>
</tbody>
</table>

</div>


<a id="7-reflection-and-references"></a>
### 7. Reflection and References

Reflection prompts:

1. Which source was easiest to load: local CSV, live CSV, or HTML table? Why?
2. What validation checks would you add before using live data in an analysis report?
3. When should a project prefer a stable downloaded dataset over live scraping?

Further readings:

- pandas `DataFrame`: <https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html>
- pandas `read_csv`: <https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html>
- Python `urllib.request`: <https://docs.python.org/3/library/urllib.request.html>
- Beautiful Soup documentation: <https://www.crummy.com/software/BeautifulSoup/bs4/doc/>
- Environment and Climate Change Canada historical climate data: <https://climate.weather.gc.ca/>
- Wikipedia: States and territories of Australia: <https://en.wikipedia.org/wiki/States_and_territories_of_Australia>
